# 03. Spike Event Analysis

This notebook covers:
1. Spike detection (absolute & percentile thresholds)
2. Event merging (consecutive intervals)
3. Pre-spike feature extraction
4. Statistical comparison: spike vs normal
5. Simple classification model

Produces:
- **Graph 4**: Monthly spike count
- **Graph 5**: Pre-spike average profile
- **Graph 6**: Spike vs non-spike feature boxplot

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src import io as data_io
from src import features
from src import models
from src import plots

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Data

In [ ]:
df = data_io.load_processed_data('nsw_processed')
df = features.compute_time_features(df)

print(f"Data: {len(df)} intervals")
print(f"Date range: {df.index.min()} to {df.index.max()}")

## 2. Spike Detection

Two definitions:
1. **Absolute**: RRP >= $300/MWh
2. **Percentile**: Top 1% of prices

In [ ]:
# Method 1: Absolute threshold
is_spike_abs = features.detect_spikes_absolute(df, threshold=config.SPIKE_ABSOLUTE_THRESHOLD)
print(f"Absolute threshold (>=${config.SPIKE_ABSOLUTE_THRESHOLD}):")
print(f"  Spike intervals: {is_spike_abs.sum():,} ({is_spike_abs.mean()*100:.2f}%)")

# Method 2: Percentile threshold
is_spike_pct, pct_threshold = features.detect_spikes_percentile(df, percentile=config.SPIKE_PERCENTILE)
print(f"\nPercentile threshold (top {100-config.SPIKE_PERCENTILE}%, >=${pct_threshold:.2f}):")
print(f"  Spike intervals: {is_spike_pct.sum():,} ({is_spike_pct.mean()*100:.2f}%)")

In [ ]:
# Use the stricter threshold for analysis (or combine)
# Here we use absolute threshold
is_spike = is_spike_abs

# Add to dataframe
df['is_spike'] = is_spike

## 3. Merge Spike Events

In [ ]:
# Merge consecutive spike intervals into events
event_ids = features.merge_spike_events(is_spike, max_gap_intervals=1)
df['event_id'] = event_ids

# Create event summary table
event_df = features.create_spike_event_table(df, event_ids)

print(f"Total spike events: {len(event_df)}")
print(f"\nEvent statistics:")
display(event_df.describe())

In [ ]:
# View top 10 most extreme events
print("Top 10 spike events by max price:")
display(event_df.nlargest(10, 'max_price'))

## 4. Graph 4: Monthly Spike Count

In [ ]:
fig = plots.plot_monthly_spike_count(event_df)
plt.show()

## 5. Pre-Spike Feature Extraction

In [ ]:
# Compute features for 30 minutes before each spike
event_df_features = features.compute_pre_spike_features(
    df, event_df, 
    window_minutes=config.PRE_SPIKE_WINDOW_MINUTES
)

print("Event features:")
display(event_df_features.head(10))

In [ ]:
# Save event table
event_df_features.to_csv(config.DATA_PROCESSED / 'spike_events.csv', index=False)
print(f"Saved event table to {config.DATA_PROCESSED / 'spike_events.csv'}")

## 6. Graph 5: Pre-Spike Average Profile

In [ ]:
fig = plots.plot_pre_spike_profile(
    df, event_df,
    window_before=60,
    window_after=30
)
plt.show()

## 7. Statistical Comparison: Spike vs Non-Spike

In [ ]:
# Define feature columns to compare
feature_cols = ['total_demand', 'hour', 'weekday']
if 'solar' in df.columns:
    feature_cols.append('solar')
if 'wind' in df.columns:
    feature_cols.append('wind')

# Compare distributions
comparison = models.compare_spike_vs_normal(df, is_spike, feature_cols)
print("Statistical comparison (spike vs normal):")
display(comparison)

## 8. Graph 6: Spike vs Non-Spike Feature Boxplot

In [ ]:
fig = plots.plot_spike_feature_comparison(
    df, is_spike,
    feature_cols=feature_cols
)
plt.show()

## 9. Spike Timing Analysis

In [ ]:
timing = models.analyze_spike_timing(event_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# By hour
ax1 = axes[0]
ax1.bar(timing['by_hour']['hour'], timing['by_hour']['count'], color='coral')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Number of Events')
ax1.set_title('Spike Events by Hour')
ax1.set_xticks(range(0, 24, 2))

# By weekday
ax2 = axes[1]
weekdays = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ax2.bar(timing['by_weekday']['weekday'], timing['by_weekday']['count'], color='steelblue')
ax2.set_xticks(range(7))
ax2.set_xticklabels(weekdays)
ax2.set_xlabel('Day of Week')
ax2.set_ylabel('Number of Events')
ax2.set_title('Spike Events by Weekday')

# By month
ax3 = axes[2]
ax3.bar(timing['by_month']['month'], timing['by_month']['count'], color='seagreen')
ax3.set_xlabel('Month')
ax3.set_ylabel('Number of Events')
ax3.set_title('Spike Events by Month')
ax3.set_xticks(range(1, 13))

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'spike_timing.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Simple Classification Model

Train a simple model to identify spike drivers (for interpretation, not prediction).

In [ ]:
# Prepare classification features
clf_features = ['hour', 'weekday', 'month']
if 'total_demand' in df.columns:
    clf_features.append('total_demand')
if 'solar' in df.columns:
    clf_features.append('solar')
if 'wind' in df.columns:
    clf_features.append('wind')

X, y = models.prepare_classification_data(df, is_spike, clf_features)
print(f"Classification data: {len(X)} samples, {len(clf_features)} features")
print(f"Class balance: {y.mean()*100:.2f}% spikes")

In [ ]:
# Fit logistic regression
try:
    lr_results = models.fit_logistic_regression(X, y)
    print("Logistic Regression Results:")
    print(f"  CV AUC: {lr_results['cv_auc_mean']:.3f} +/- {lr_results['cv_auc_std']:.3f}")
    print(f"\nFeature coefficients (sorted by importance):")
    display(lr_results['coefficients'])
except Exception as e:
    print(f"Could not fit logistic regression: {e}")

In [ ]:
# Fit decision tree for interpretability
try:
    dt_results = models.fit_decision_tree(X, y, max_depth=4)
    print("Decision Tree Results:")
    print(f"  CV AUC: {dt_results['cv_auc_mean']:.3f} +/- {dt_results['cv_auc_std']:.3f}")
    print(f"\nFeature importance:")
    display(dt_results['feature_importance'])
except Exception as e:
    print(f"Could not fit decision tree: {e}")

## 11. Key Findings

Summarize your findings:

- **When do spikes occur?** (hours, days, months)
- **What precedes spikes?** (demand changes, renewable changes)
- **Top drivers identified by model:**

---
## Next Steps
Proceed to `04_volatility.ipynb` for volatility analysis.